# LAPORAN TUGAS AKHIR: SISTEM DETEKSI DAN PERINGATAN PELANGGARAN ALAT PELINDUNG DIRI MENGGUNAKAN EDGE DEVICE BERBASIS YOLO DALAM KAWASAN KONSTRUKSI
**Fokus Bahasan: Proses Prapemrosesan Dataset & Pelatihan Model Deteksi APD Pekerja**

---

## DESKRIPSI DATASET & DESAIN KELAS
Model deteksi Alat Pelindung Diri (APD) pada Tugas Akhir ini dirancang menggunakan arsitektur **YOLOv8** berbasis regresi koordinat bounding box. Dataset terdiri dari total **1.625 gambar asli** yang dibagi menjadi data training (80%), validation (10%), dan testing (10%).

Sistem deteksi dirancang menggunakan **6 Kelas Klasifikasi APD**:
1.  `helmet` (Pekerja memakai helm keselamatan - patuh)
2.  `no_helmet` (Pekerja tidak memakai helm keselamatan - melanggar)
3.  `vest` (Pekerja memakai rompi keselamatan - patuh)
4.  `no_vest` (Pekerja tidak memakai rompi keselamatan - melanggar)
5.  `safety-shoes` (Pekerja memakai sepatu safety - patuh)
6.  `no_safety-shoes` (Pekerja tidak memakai sepatu safety - melanggar)

Notebook ini mendokumentasikan langkah prapemrosesan dataset, pembuatan gambar potongan tubuh pekerja (*cropped person*), dan jalannya proses training model.

In [ ]:
import os
import sys
from pathlib import Path

# Daftarkan root direktori proyek ke sys.path untuk impor modul lokal
project_dir = Path(os.getcwd())
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.train import check_dataset_and_classes, prepare_cropped_dataset, run_training
print("Modul prapemrosesan & pelatihan dimuat!")

## FASE 1: VERIFIKASI STRUKTUR DATASET AWAL
Sebelum prapemrosesan dijalankan, kita perlu memverifikasi kesiapan berkas konfigurasi metadata dataset asli (`data.yaml`) dan melacak jumlah kelas label yang terdaftar.

In [ ]:
print("[Proses] Memverifikasi dataset...")
check_dataset_and_classes()

## FASE 2: PREPROCESSING DATASET DUA-TAHAP (CROPPING PEKERJA)
Untuk memfokuskan deteksi APD, sistem menggunakan pendekatan dua-tahap:
1.  Mencari lokasi objek `person` pada gambar asli menggunakan model detektor awal (`best_person.pt`).
2.  Memotong gambar koordinat tubuh pekerja tersebut (*cropped person*) dengan memberikan batas *padding bawah* aman (agar kaki/sepatu tidak terpotong).
3.  Menyimpan potongan koordinat tersebut ke folder `assets/dataset_cropped/` sebagai data latih model deteksi APD utama.

### ANALISIS PERUBAHAN FASE REBALANCING & OVERSAMPLING DATASET
Dalam penyusunan Tugas Akhir ini, terjadi fase transisi penting terkait manipulasi pengulangan data (*oversampling*) akibat diterapkannya pembersihan data (*data cleaning*):

1.  **Kendala Dataset Awal (Ketimpangan Kelas)**:
    Sebelumnya, dataset mentah memiliki ketidakseimbangan kelas (*class imbalance*) yang mencolok. Untuk mengatasinya secara manual, program menerapkan faktor pengulangan lokal (`oversample_factor > 1`) guna melipatgandakan data kelas minoritas saat proses cropping.
2.  **Fase Data Cleaning & Rebalancing**:
    Kami melakukan pembersihan data secara menyeluruh untuk memilah citra berkualitas tinggi dan memperbaiki anotasi yang rusak. Dataset yang baru diseimbangkan ulang menggunakan augmentasi terintegrasi dari platform Roboflow, menghasilkan dataset final sebanyak **1.500 data latih (train), 63 validasi (val), dan 62 pengujian (test)**.
3.  **Keputusan Menonaktifkan Oversampling Lokal (`oversample_factor = 1`)**:
    Karena dataset hasil ekspor Roboflow sudah dalam kondisi bersih dan telah teraugmentasi secara matang (*pre-augmented*), pemakaian oversampling lokal tambahan dinonaktifkan (`oversample_factor` disetel ke `1`). Hal ini dilakukan demi:
    *   **Mencegah Double-Augmentation**: Mencegah model mengalami *overfitting* akut akibat melatih gambar dengan varian manipulasi yang sama secara berulang-ulang.
    *   **Efisiensi Penyimpanan**: Mengurangi beban pembentukan data sampah pada disk penyimpanan Jetson Nano.

Jalankan cell di bawah untuk memproses cropping otomatis:

In [ ]:
print("[Proses] Memulai pembuatan dataset cropped...")
prepare_cropped_dataset(force_recreate=False)

## FASE 3: TRAINING MODEL UTAMA APD (YOLOV8)
Pelatihan model utama (Stage 2) dijalankan menggunakan dataset yang sudah dicrop. Kita menggunakan model dasar pretrained `yolov8n.pt` dan hyperparameter optimal (`batch=16`, `imgsz=640`, mixed precision `amp=True`, dan pembekuan layer backbone `freeze=10`).

Jalankan cell di bawah untuk melatih model APD Anda:

In [ ]:
# Jalankan proses pelatihan model APD (Stage 2)
# Untuk simulasi uji coba cepat, kita jalankan 5 epoch terlebih dahulu
EPOCHS_TEST = 5
print(f"[Training] Memulai pelatihan model APD untuk {EPOCHS_TEST} epoch...")
run_training(epochs=EPOCHS_TEST, resume=False)

## HASIL EVALUASI & METRIK AKURASI MODEL
Setelah proses training selesai, visualisasi grafik performa model akan disimpan secara otomatis di direktori `runs/detect/train/`:
*   **`results.png`**: Grafik penurunan loss (Box, Class, DFL) dan kenaikan metrik akurasi mAP50 secara periodik.
*   **`confusion_matrix_normalized.png`**: Matriks untuk mengukur seberapa akurat model membedakan kelas patuh (`helmet`, `vest`, `safety shoes`) dan kelas melanggar (`no-helmet`, `no-vest`, `no-safety shoes`).
*   **Metrik Akhir**: Hasil akurasi akhir dapat dibaca melalui file CSV `runs/detect/train/results.csv` untuk kemudian dianalisis dalam bab evaluasi dokumen Tugas Akhir.